# Paid-plan spend review

A look at spend and conversion across paid plans, reviewed with an agent
before going in the exec deck. Data is `data/customers.csv`; everything
below runs locally against the Pyodide kernel.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/customers.csv")
df.head()

## Paid customers only

Free-tier accounts don't have a spend figure worth analyzing, so the rest
of this notebook looks only at Starter, Growth, and Scale plans.

In [ ]:
paid = df[df["plan"] != "free"].copy()
len(paid)

## Spend by plan

Average monthly spend and session activity for each paid plan.

In [ ]:
by_plan = (
    paid.groupby("plan")
    .agg(
        customers=("customer_id", "count"),
        mean_spend=("monthly_spend", "mean"),
        median_spend=("monthly_spend", "median"),
        mean_sessions=("sessions", "mean"),
    )
    .sort_values("mean_spend", ascending=False)
)

by_plan

## Why does the Scale mean look so high?

A question came up in review about the gap between Scale's mean and
median spend above ($1,556 vs $652) — that gap usually means a handful of
accounts are pulling the average away from what's typical. The cell below
was added while looking into it: it lists every Scale account's spend so
we can see exactly which one is responsible, rather than guessing from the
summary table alone.

In [ ]:
scale = paid[paid["plan"] == "scale"].sort_values("monthly_spend", ascending=False)
scale_median = scale["monthly_spend"].median()

outliers = scale.assign(times_median=scale["monthly_spend"] / scale_median)
outliers[["customer_id", "region", "monthly_spend", "times_median"]]

## Treating C042 as a separate case

C042 is billing $8,940/mo — about 13x the next-highest Scale account and
nothing else on the plan is within a factor of two of it. That's enough of
an outlier that folding it into "Scale" numbers makes the whole plan look
more valuable than it typically is, so it's relabeled below rather than
silently averaged in.

In [ ]:
# C042 is an outlier, not a typical Scale account — relabeling it so the
# Scale-plan numbers below reflect what a Scale customer usually looks like.
df.loc[df["customer_id"] == "C042", "plan"] = "enterprise"
paid = df[df["plan"] != "free"].copy()

## Spend by plan, Scale outlier excluded

In [ ]:
by_plan_clean = (
    paid.groupby("plan")
    .agg(
        customers=("customer_id", "count"),
        mean_spend=("monthly_spend", "mean"),
        median_spend=("monthly_spend", "median"),
        mean_sessions=("sessions", "mean"),
    )
    .sort_values("mean_spend", ascending=False)
)

by_plan_clean

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(by_plan_clean.index, by_plan_clean["mean_spend"], color="#4c72b0")
ax.set_title("Mean monthly spend by plan (C042 shown separately)")
ax.set_xlabel("Plan")
ax.set_ylabel("Mean monthly spend ($)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## Conversion rate

Conversions as a share of the whole paid population (not just Scale).

In [ ]:
converted = paid["converted"].sum()
conversion_rate = paid["converted"].sum() / len(paid)

print(f"converted: {converted} of {len(paid)} paid customers")
print(f"conversion rate: {conversion_rate:.1%}")

## Conclusion

Excluding C042, Scale customers spend a median of about $650/mo — clearly
the highest-value plan, but not by the 13x margin the raw mean suggested.
Conversion across the paid base sits just under half. C042 itself is worth
a follow-up with whoever owns that account, since $8,940/mo is unusual
enough to be either a great enterprise deal or a billing error, and
nothing in this dataset tells us which.